In [ ]:
# This notebook can be used to train a model to predict translation features from English to Spanish, or Spanish to English

In [ ]:
# Install dependencies
!pip install -r ../requirements.txt 

In [ ]:
!pip install torchcodec

In [ ]:
!pip install librosa

In [ ]:
!pip install transformers

In [ ]:
!pip install umap-learn

In [ ]:
# Instalar librosa
import sys
import subprocess
import pkg_resources

required = {'librosa'}
installed = {pkg.key for pkg in pkg_resources.working_set}
missing = required - installed

if missing:
    print(f"Instalando {missing}...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print("librosa ya está instalado")

# Reiniciar kernel
print("Por favor, reinicia el kernel manualmente (Kernel → Restart)")

In [ ]:
# Instalar librosa
import sys
import subprocess
import pkg_resources

required = {'transformers'}
installed = {pkg.key for pkg in pkg_resources.working_set}
missing = required - installed

if missing:
    print(f"Instalando {missing}...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print("transformers ya está instalado")

# Reiniciar kernel
print("Por favor, reinicia el kernel manualmente (Kernel → Restart)")

In [ ]:
# Instalar librosa
import sys
import subprocess
import pkg_resources

required = {'umap-learn'}
installed = {pkg.key for pkg in pkg_resources.working_set}
missing = required - installed

if missing:
    print(f"Instalando {missing}...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print("umap-learn ya está instalado")

# Reiniciar kernel
print("Por favor, reinicia el kernel manualmente (Kernel → Restart)")

In [ ]:
import sys
import os
import random
import pickle
import torch.nn as nn
import torch

# Add parent directory to Python path
sys.path.append(os.path.abspath('..'))

import prepare_dataset as pd
import feature_selection as fs
import train_translation_model as ttm

In [ ]:
# Obtain DRAL data and put in the right place
# Running this cell is optional
# If the data is already present in the target directory, by default the data will not be downloaded again.

import os, tarfile, shutil, requests
from tqdm.notebook import tqdm

# Paths
data_dir = "../data"
dral_dir = os.path.join(data_dir, "dral")
target_loc = os.path.join(dral_dir, "fragments-short")
url = "https://www.cs.utep.edu/nigel/dral/DRAL-16kHz.tgz"
tgz_path = os.path.join(data_dir, "DRAL-16kHz.tgz")

if not os.path.exists(target_loc):
    os.makedirs(dral_dir, exist_ok=True)

    print("Downloading DRAL dataset...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total_size = int(r.headers.get('content-length', 0))
        block_size = 1024 * 1024
        with open(tgz_path, 'wb') as f, tqdm(
            total=total_size, unit='B', unit_scale=True, desc="Downloading"
        ) as pbar:
            for chunk in r.iter_content(chunk_size=block_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    print("Extracting selected folders...")
    with tarfile.open(tgz_path, "r:gz") as tar:
        wanted = {"fragments-short", "metadata"}
        members = [
            m for m in tar.getmembers()
            if m.name.startswith("DRAL16kHz/")
            and len(m.name.split("/")) > 1
            and m.name.split("/")[1] in wanted
        ]
        tar.extractall(path=data_dir, members=members)

    # Move wanted folders into data/dral/
    extracted_root = os.path.join(data_dir, "DRAL16kHz")
    for folder in wanted:
        src = os.path.join(extracted_root, folder)
        dst = os.path.join(dral_dir, folder)
        if os.path.exists(src):
            shutil.move(src, dst)

    # Cleanup
    os.remove(tgz_path)
    shutil.rmtree(extracted_root, ignore_errors=True)

    print("DRAL data ready in:", os.path.abspath(dral_dir))
else:
    print("DRAL data already present at:", os.path.abspath(dral_dir))


In [ ]:
# Prepare all the features. This step may take a few hours the first time. Afterwards, features are not computed again if already present.
# might benefit from some sort of multiprocessing (not yet implemented)

en_train = pd.load_filelist(os.path.join(data_dir, "filelists/train_filelist_en.txt"))
es_train = pd.load_filelist(os.path.join(data_dir, "filelists/train_filelist_es.txt"))
en_test = pd.load_filelist(os.path.join(data_dir, "filelists/test_filelist_en.txt"))
es_test = pd.load_filelist(os.path.join(data_dir, "filelists/test_filelist_es.txt"))

feature_directory = os.path.join(data_dir, "features1")
if not os.path.exists(feature_directory):
    os.makedirs(feature_directory, exist_ok=True)

# save features in folder
pd.extract_and_save_features(en_train, target_loc, feature_directory)
pd.extract_and_save_features(es_train, target_loc, feature_directory)
pd.extract_and_save_features(en_test, target_loc, feature_directory)
pd.extract_and_save_features(es_test, target_loc, feature_directory)



In [ ]:
data_dir = "../data"
pred_dir = os.path.join(data_dir, "spanish_test")
feature_dir = feature_directory = os.path.join(data_dir, "english")

In [ ]:
pd.process_challenge_data(pred_dir, feature_dir)

In [ ]:
pd.process_with_pca(pred_dir, feature_dir)

In [ ]:
pd.process_with_entropy_selection(pred_dir, feature_dir)

In [ ]:
pd.process_challenge_data_transf(pred_dir, feature_dir)

In [ ]:
pd.process_with_UMAP(pred_dir, feature_dir)

In [ ]:
pd.process_with_KBest(pred_dir, feature_dir)

In [ ]:
pd.process_with_Centroid(pred_dir, feature_dir)

In [ ]:
data_dir = "../data"
pred_dir = os.path.join(data_dir, "english_test")
feature_dir = feature_directory = os.path.join(data_dir, "spanish)

In [ ]:
pd.process_challenge_data1(pred_dir, feature_dir)

In [ ]:
pd.process_with_pca1(pred_dir, feature_dir)

In [ ]:
pd.process_with_entropy_selection1(pred_dir, feature_dir)

In [ ]:
pd.process_challenge_data_transf1(pred_dir, feature_dir)

In [ ]:
pd.process_with_UMAP1(pred_dir, feature_dir)

In [ ]:
pd.process_with_KBest1(pred_dir, feature_dir)

In [ ]:
pd.process_with_Centroid1(pred_dir, feature_dir)

In [ ]:
import os
import numpy as np

folder = feature_dir 
for file in os.listdir(folder):
    if file.endswith('.npy'):
        try:
            data = np.load(os.path.join(folder, file))
            print(data)
            if data.shape != (103,):
                print(f"❌ ERROR: {file} tiene forma {data.shape}")
            # Opcional: chequear si hay valores NaN
            if np.isnan(data).any():
                print(f"⚠️ AVISO: {file} contiene NaNs")
        except Exception as e:
            print(f"🚨 ARCHIVO CORRUPTO: {file} - {e}")

print("Validación terminada.")

In [ ]:
# load averaged features computed in the previous steps (faster than obtaining the features through 'on-the-spot' computation)
avgd_features_en_train = ttm.load_model_features(en_train, feature_directory)
avgd_features_es_train = ttm.load_model_features(es_train, feature_directory)
avgd_features_en_test = ttm.load_model_features(en_test, feature_directory)
avgd_features_es_test = ttm.load_model_features(es_test, feature_directory)

In [ ]:
# Run model for English to Spanish. Should take less than a minute to train.
# For English, we retain the 1024-dim features, while for Spanish we extract the 101 most pragmatically-salient features.
avgd_selected_features_es_train = []
for train_features_to_reduce in avgd_features_es_train:
    avgd_selected_features_es_train.append(fs.extract_winners(train_features_to_reduce, "spanish"))
   
    
avgd_selected_features_es_test = []
for test_features_to_reduce in avgd_features_es_test:
    avgd_selected_features_es_test.append(fs.extract_winners(test_features_to_reduce, "spanish"))
    

# Model is a MLP, learning a mapping from 1024 final-layer averaged English features to 101 selected final-layer averaged Spanish features
en_to_es_mlp_model = ttm.MLP(layers=[(1024,500), (500,250), (250,125)], output=(125,101), hidden_activation=nn.ReLU)
mae, mse, cosine_sim = ttm.train_test_model(model=en_to_es_mlp_model, X_train=avgd_features_en_train, X_test=avgd_features_en_test, y_train=avgd_selected_features_es_train, y_test=avgd_selected_features_es_test, epochs=40, lr=0.001, batch_size=500)
print("Results for 1024f -> Language Subset | Eng -> Spa:")
print(f"MAE: {mae} \nMSE: {mse} \nCosine Similarity: {cosine_sim}")
print("-"*40)

if not os.path.exists("../Models"):
    os.makedirs("../Models")
    
#Saving model
model_filename = '../Models/english_to_spanish_model.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(en_to_es_mlp_model, file)
print(f"Model saved to {model_filename}")

In [ ]:
avgd_features_en_test

In [ ]:
# Run model for Spanish to English. Should need less than a minute to train.
# For Spanish, we retain the 1024-dim features, while for English we extract the 103 most pragmatically-salient features.

avgd_selected_features_en_train = []
for train_features_to_reduce in avgd_features_en_train:
    avgd_selected_features_en_train.append(fs.extract_winners(train_features_to_reduce, "english"))

avgd_selected_features_en_test = []
for test_features_to_reduce in avgd_features_en_test:
    avgd_selected_features_en_test.append(fs.extract_winners(test_features_to_reduce, "english"))

# Model is a MLP, learning a mapping from 1024 final-layer averaged Spanish features to 103 selected final-layer averaged English features
es_to_en_mlp_model = ttm.MLP(layers=[(1024,500), (500,250), (250,125)], output=(125,103), hidden_activation=nn.ReLU)
mae, mse, cosine_sim = ttm.train_test_model(model=es_to_en_mlp_model, X_train=avgd_features_es_train, X_test=avgd_features_es_test, y_train=avgd_selected_features_en_train, y_test=avgd_selected_features_en_test, epochs=40, lr=0.001, batch_size=500)
print("Results for 1024f -> Language Subset | Spa -> Eng:")
print(f"MAE: {mae} \nMSE: {mse} \nCosine Similarity: {cosine_sim}")
print("-"*40)

if not os.path.exists("../Models"):
    os.makedirs("../Models")
    
#Saving model
model_filename = '../Models/spanish_to_english_model.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(es_to_en_mlp_model, file)
print(f"Model saved to {model_filename}")

In [ ]:
dral_dir = os.path.join(data_dir, "dral")
target_loc = os.path.join(dral_dir, "fragments-short")